In [1]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

c:\Users\knguy139\Documents\Projects\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
# Connect to Snowflake
engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

In [3]:
# hospital_group aggregation — MNR FFS, 2025 only (202501-202512)
df = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count_2025
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt_2025
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
        and d.collection is not null
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

print(f"Core (2025): {df.shape}")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVLRbtowFP2VyHtO7GQQwAKqDFoNqe0Y0An1ZXJsB7w6dmY7BPb1cwJI7UMr9cGSZZ9zz7n33PHNsZTBgRsrtJqAOEIg4IpqJtRuAp42d%2BEQBNYRxYjUik%2FAiVtwMx1bUsoKZ7XbqxX%2FW3PrAl9IWdx%2BTEBtFNbECosVKbnFjuJ19nCPkwhhYi03zsuBC4VZ4bX2zlUYwqZpouZrpM0OJgghiEbQo1rIF%2FBKovpYozLaaarllXL0Pb0jEUPUayU8wissL8RvQp1H8JFKfgZZ%2FH2zWYbLH%2BsNCLJrdzOtbF1ys%2BbmICh%2FWt2fDVjvoN7vQn9YQ4j9vSeRVbopJHnhVJdV7XzNyN9gwRmUeif8pBbzCaheBNv%2BfFiPBs1gezodZo1eZcvbP%2Btj%2FKyb9JZu%2BayXZ6LYP%2F5DB01B8Ouaa9LmurC25gvVpun8E0rSEKVhnG7iGCcpRqOoPxg%2Bg2Du0xSKuI55tdz5iEpBjba6cFpJoXjnkuWoXxBKQjpMSNjLRyzMR7QfoiLt5emg3%2B8lMWwzS8B5b3BnxEw%2FN40xfM29LOCjz2QxX2op6Cm406Yk7v3I4ijuXgQLiw6KeUmEzBgz3FofnZS6mRlOnN9zZ2oO4PSs%2BnbTp%2F8B&RelayState=ver%3A3-hint%3A260092117886646-ETMsDgAAAZ7QLoKqABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEG%2BEcLA8jhk6LCEiBiH0DKQAAACQJ8xILt1g

In [4]:
# member_appeal — 202601+ only (separate time range)
df_ma = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count_2026
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt_2026
        , count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt
        , count(distinct case when a.member_appeal_ovtn_ind = 1 then a.case_id end) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
        and d.collection is not null
    group by 1,2
    having count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) >= 10
""", engine)

engine.dispose()
print(f"Member appeal (202601+): {df_ma.shape}")

Member appeal (202601+): (918, 6)


In [5]:
# merge and compute rates
df = (
    df
    .merge(df_ma, on = ["hospital_group", "fin_market"], how = "left")
    .assign(
        initial_adr_rate_2025 = lambda x: x.initial_adr_cnt_2025 / x.case_count_2025,
        persistent_adr_rate = lambda x: x.persistent_adr_cnt / x.case_count_2025,
        p2p_overturn_rate = lambda x: x.p2p_ovrtn_cnt / x.initial_adr_cnt_2025.replace(0, np.nan),
        appeal_overturn_rate = lambda x: x.appeal_ovrtn_cnt / x.initial_adr_cnt_2025.replace(0, np.nan),
        mcr_overturn_rate = lambda x: x.mcr_ovrtn_cnt / x.initial_adr_cnt_2025.replace(0, np.nan),
        md_review_rate = lambda x: x.md_reviewed_cnt / x.case_count_2025,
        member_appeal_rate = lambda x: x.member_appeal_cnt / x.initial_adr_cnt_2026.replace(0, np.nan),
        initial_adr_rate_2026 = lambda x: x.initial_adr_cnt_2026 / x.case_count_2026,
    )
    .dropna(subset = ["p2p_overturn_rate", "member_appeal_rate"])
)

rate_cols = [
    "initial_adr_rate_2025", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
    "member_appeal_rate", "initial_adr_rate_2026"
]
count_cols = [
    "case_count_2025", "initial_adr_cnt_2025", "persistent_adr_cnt", "p2p_ovrtn_cnt",
    "appeal_ovrtn_cnt", "mcr_ovrtn_cnt", "md_reviewed_cnt",
    "case_count_2026", "member_appeal_cnt", "initial_adr_cnt_2026"
]
# Change null to 0
df[rate_cols + count_cols] = df[rate_cols + count_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Shape: {df.shape[0]}")

Shape: 904


In [6]:
# =============================================================================
# DATA PREP: Eligibility filter + Empirical Bayes shrinkage + log-volume features
# =============================================================================
from scipy.stats import median_abs_deviation

# Split a new df out for IF analysis
df_iso = df.copy()
df_iso[count_cols] = df_iso[count_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# --- Eligibility filter: minimum volume thresholds ---
# Ensures flagged entities have enough sample size to be meaningful
df_iso = (
    df_iso
    .query("case_count_2025 >= 30 and initial_adr_cnt_2025 >= 10 and initial_adr_cnt_2026 >= 10")
    .copy()
)

# --- Empirical Bayes shrinkage (credibility weighting) ---
# adjusted_rate = (observed_num + k * global_rate) / (observed_den + k)
# k=50 pulls small-sample rates toward the population mean to reduce noise
# Hospitals with few cases get pulled more toward the global average
# Hospitals with many cases stay close to their observed rate.
k = 50

# global rates (population-level averages across all eligible hospitals)
gr_initial_adr_25 = df_iso["initial_adr_cnt_2025"].sum() / df_iso["case_count_2025"].sum()
gr_persistent = df_iso["persistent_adr_cnt"].sum() / df_iso["case_count_2025"].sum()
gr_p2p = df_iso["p2p_ovrtn_cnt"].sum() / df_iso["initial_adr_cnt_2025"].sum()
gr_appeal = df_iso["appeal_ovrtn_cnt"].sum() / df_iso["initial_adr_cnt_2025"].sum()
gr_mcr = df_iso["mcr_ovrtn_cnt"].sum() / df_iso["initial_adr_cnt_2025"].sum()
gr_md = df_iso["md_reviewed_cnt"].sum() / df_iso["case_count_2025"].sum()
gr_member_appeal = df_iso["member_appeal_cnt"].sum() / df_iso["initial_adr_cnt_2026"].sum()
gr_initial_adr_26 = df_iso["initial_adr_cnt_2026"].sum() / df_iso["case_count_2026"].sum()

# shrinkage-adjusted rates (overwrite the raw rates with stabilized versions)
df_iso = df_iso.assign(
    initial_adr_rate_2025 = lambda x: (x.initial_adr_cnt_2025 + k * gr_initial_adr_25) / (x.case_count_2025 + k),
    persistent_adr_rate = lambda x: (x.persistent_adr_cnt + k * gr_persistent) / (x.case_count_2025 + k),
    p2p_overturn_rate = lambda x: (x.p2p_ovrtn_cnt + k * gr_p2p) / (x.initial_adr_cnt_2025 + k),
    appeal_overturn_rate = lambda x: (x.appeal_ovrtn_cnt + k * gr_appeal) / (x.initial_adr_cnt_2025 + k),
    mcr_overturn_rate = lambda x: (x.mcr_ovrtn_cnt + k * gr_mcr) / (x.initial_adr_cnt_2025 + k),
    md_review_rate = lambda x: (x.md_reviewed_cnt + k * gr_md) / (x.case_count_2025 + k),
    member_appeal_rate = lambda x: (x.member_appeal_cnt + k * gr_member_appeal) / (x.initial_adr_cnt_2026 + k),
    initial_adr_rate_2026 = lambda x: (x.initial_adr_cnt_2026 + k * gr_initial_adr_26) / (x.case_count_2026 + k),
)

# --- Log-volume features ---
# Gives IF explicit volume signal so it can distinguish
# probable outlier vs "outlier due to small sample"
df_iso = df_iso.assign(
    log_case_count_2025 = lambda x: np.log1p(x.case_count_2025),
    log_initial_adr_cnt_2025 = lambda x: np.log1p(x.initial_adr_cnt_2025),
    log_initial_adr_cnt_2026 = lambda x: np.log1p(x.initial_adr_cnt_2026),
)

# --- Features going into Isolation Forest ---
iso_kpi = [
    "log_case_count_2025", "log_initial_adr_cnt_2025", "log_initial_adr_cnt_2026",
    "initial_adr_rate_2025", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
    "member_appeal_rate", "initial_adr_rate_2026"
]

df_iso[iso_kpi] = df_iso[iso_kpi].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"IF-ready: {df_iso.shape[0]} hospitals, {len(iso_kpi)} features")

IF-ready: 893 hospitals, 11 features


In [7]:
# =============================================================================
# ISOLATION FOREST: anomaly detection
# - Distribution-free; contamination=5% sets expected outlier proportion
# - 300 trees for stable scoring
# - StandardScaler normalizes features so no single metric dominates
# =============================================================================
scaler_iso = StandardScaler()
X_iso = scaler_iso.fit_transform(df_iso[iso_kpi])

iso_model = IsolationForest(contamination = 0.05, n_estimators = 300, random_state = 42)
iso_model.fit(X_iso)

df_iso = df_iso.assign(
    anomaly_score = iso_model.decision_function(X_iso),
    anomaly_flag = iso_model.predict(X_iso) == -1,
)
df_iso["anomaly_rank"] = df_iso["anomaly_score"].rank(method = "dense", ascending = True).astype(int)

print(f"Flagged: {df_iso['anomaly_flag'].sum()} / {df_iso.shape[0]} hospitals")

Flagged: 45 / 893 hospitals


In [8]:
# =============================================================================
# MAD Z-SCORES (interpretability layer)
# - MAD z = (x - median) / (MAD * 1.4826)
# - 1.4826 = consistency constant making MAD comparable to SD under normality
# - Used for driver identification, NOT for IF model input
# - More robust to skew than standard z-scores
# =============================================================================
mad_z_cols = [f"{c}_z" for c in iso_kpi]

for col in iso_kpi:
    med = df_iso[col].median()
    mad_scaled = median_abs_deviation(df_iso[col], scale = 1.0) * 1.4826
    df_iso[f"{col}_z"] = (df_iso[col] - med) / mad_scaled if mad_scaled > 0 else 0

print(f"MAD z-scores computed for {len(iso_kpi)} features")

MAD z-scores computed for 11 features


In [9]:
# =============================================================================
# DRIVER IDENTIFICATION
# - For each row: which MAD z-score has the largest absolute value?
# - That metric is the primary "reason" this hospital was flagged.
# =============================================================================
df_iso = df_iso.assign(
    driver = lambda x: x[mad_z_cols].abs().idxmax(axis = 1).str.removesuffix("_z"),
    z_score = lambda x: x[mad_z_cols].apply(lambda r: r[r.abs().idxmax()], axis = 1),
    direction = lambda x: np.where(x.z_score > 0, "high", "low"),
    reason = lambda x: np.where(x.anomaly_flag, x.driver + " (" + x.direction + ")", ""),
)

# --- Driver value, population median, and % deviation ---
# For log-transformed drivers, we report the raw count instead of the log value
# so the output is human-readable (e.g. "case_count = 500" not "log = 6.2")
pop_medians = df_iso[iso_kpi].median()
raw_med_case_count_2025 = df_iso["case_count_2025"].median()
raw_med_initial_adr_2025 = df_iso["initial_adr_cnt_2025"].median()
raw_med_initial_adr_2026 = df_iso["initial_adr_cnt_2026"].median()

df_iso["driver_value"] = np.nan
df_iso["driver_pop_median"] = np.nan
df_iso["driver_dev"] = np.nan

# log drivers → use raw values
mask = df_iso["anomaly_flag"] & (df_iso["driver"] == "log_case_count_2025")
df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, "case_count_2025"]
df_iso.loc[mask, "driver_pop_median"] = raw_med_case_count_2025
df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, "case_count_2025"] - raw_med_case_count_2025) / raw_med_case_count_2025

mask = df_iso["anomaly_flag"] & (df_iso["driver"] == "log_initial_adr_cnt_2025")
df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, "initial_adr_cnt_2025"]
df_iso.loc[mask, "driver_pop_median"] = raw_med_initial_adr_2025
df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, "initial_adr_cnt_2025"] - raw_med_initial_adr_2025) / raw_med_initial_adr_2025

mask = df_iso["anomaly_flag"] & (df_iso["driver"] == "log_initial_adr_cnt_2026")
df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, "initial_adr_cnt_2026"]
df_iso.loc[mask, "driver_pop_median"] = raw_med_initial_adr_2026
df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, "initial_adr_cnt_2026"] - raw_med_initial_adr_2026) / raw_med_initial_adr_2026

# rate drivers → use the rate value directly
for rate_col in rate_cols:
    mask = df_iso["anomaly_flag"] & (df_iso["driver"] == rate_col)
    if mask.any():
        df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, rate_col]
        df_iso.loc[mask, "driver_pop_median"] = pop_medians[rate_col]
        df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, rate_col] - pop_medians[rate_col]) / pop_medians[rate_col]

print(f"Drivers assigned for {df_iso['anomaly_flag'].sum()} flagged hospitals")

Drivers assigned for 45 flagged hospitals


In [10]:
# =============================================================================
# OUTPUT ASSEMBLY
# =============================================================================
output_cols = [
    "hospital_group", "fin_market",
    "case_count_2025", "initial_adr_cnt_2025", "case_count_2026", "initial_adr_cnt_2026",
    *iso_kpi, *mad_z_cols,
    "anomaly_score", "anomaly_rank", "anomaly_flag",
    "driver", "z_score", "direction", "reason",
    "driver_value", "driver_pop_median", "driver_dev"
]

df_iso_output = df_iso[output_cols].sort_values("anomaly_score").reset_index(drop = True)

print(f"Model input: {df_iso.shape[0]} entities | Flagged: {df_iso['anomaly_flag'].sum()}")
df_iso_output

Model input: 893 entities | Flagged: 45


,hospital_group,fin_market,case_count_2025,initial_adr_cnt_2025,case_count_2026,initial_adr_cnt_2026,log_case_count_2025,log_initial_adr_cnt_2025,log_initial_adr_cnt_2026,initial_adr_rate_2025,...,anomaly_score,anomaly_rank,anomaly_flag,driver,z_score,direction,reason,driver_value,driver_pop_median,driver_dev
0,ADVOCATE HEALTH & HOSPITALS IL,IL,7393,78,3403.0,34.0,8.908424,4.369448,3.555348,0.012433,...,-0.097583,1,True,md_review_rate,-7.661289,low,md_review_rate (low),0.027089,0.695613,-0.961058
1,Stanford Health Hospital,CA-N,753,62,326.0,35.0,6.625392,4.143135,3.583519,0.095317,...,-0.093923,2,True,appeal_overturn_rate,13.826097,high,appeal_overturn_rate (high),0.056839,0.005154,10.028078
2,WESTCHESTER GENERAL HOSPITAL,FL,114,79,51.0,42.0,4.744932,4.382027,3.761200,0.570364,...,-0.087593,3,True,member_appeal_rate,6.967181,high,member_appeal_rate (high),0.082776,0.020448,3.048062
3,Atrium Health System,NC,15852,4912,6070.0,1958.0,9.671114,8.499640,7.580189,0.309806,...,-0.083692,4,True,member_appeal_rate,71.732105,high,member_appeal_rate (high),0.662159,0.020448,31.381973
4,Nassau Health,NY,128,75,46.0,26.0,4.859812,4.330733,3.295837,0.503032,...,-0.067774,5,True,member_appeal_rate,17.740116,high,member_appeal_rate (high),0.179150,0.020448,7.761097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
888,Community Health Systems,NM,184,44,76.0,25.0,5.220356,3.806662,3.258097,0.250169,...,0.153610,889,False,log_initial_adr_cnt_2025,-0.627046,low,,NaN,NaN,NaN
889,INSPIRA MEDICAL CENTERS,NJ,385,117,157.0,47.0,5.955837,4.770685,3.871201,0.302390,...,0.157802,890,False,appeal_overturn_rate,-0.792568,low,,NaN,NaN,NaN
890,SOUTHEAST MO HOSP ASSOCIATION,MO,422,129,242.0,80.0,6.047372,4.867534,4.394449,0.304109,...,0.157809,891,False,appeal_overturn_rate,-0.831865,low,,NaN,NaN,NaN
891,CONWAY HOSPITAL,SC,730,208,204.0,66.0,6.594413,5.342334,4.204693,0.285307,...,0.158201,892,False,log_initial_adr_cnt_2025,0.428995,high,,NaN,NaN,NaN


In [11]:
# =============================================================================
# WRITE FLAGGED HOSPITALS TO SNOWFLAKE
# =============================================================================
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine_write = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

flagged = df_iso_output[df_iso_output["anomaly_flag"] == True].copy()
flagged.columns = [c.upper() for c in flagged.columns]

flagged.to_sql(
    "kn_loc_if_outlier_hospitals_mnr",
    engine_write,
    schema = "TMP_1M",
    if_exists = "replace",
    index = False
)

print(f"Wrote {len(flagged)} flagged hospitals to TMP_1M.KN_LOC_IF_OUTLIER_HOSPITALS_MNR")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVLRbqMwEPwV5HsGDAFKrCRVWq5qru1dlNDq1JeTMSaxCjbnNaH9%2BxqSSL2HVroHS5Y9szO7s7PL16Z2DlyDUHKOAg8jh0umSiF3c%2FSY37gpcsBQWdJaST5HbxzQ5WIGtKlbsuzMXm74346DcWwhCWT4mKNOS6IoCCCSNhyIYWS7fLgnoYcJBeDaWDl0opQgrNbemJb4ft%2F3Xj%2FxlN75IcbYx1PfogbIN%2FRBov1ao9XKKKbqM%2BXV9vSJRODjaJCwCKuwPhGvhDyO4CuV4ggCcpvna3f9a5sjZ3nu7lpJ6Bqut1wfBOOPm%2FujAbAOuv3OtafsKYU%2Fe%2BqBVH1V0xfOVNN2xtb07M2veOnXaifspFbZHLUvolzfRjdtNGmKhu42h2eTZT%2BCuDBXD5PsqbtjNL37nUL6Pc5NypDzdM41HHJdAXR8JYc0jX3CYeLixA2SPAhImJAw9jC%2BeEZOZtMUkpqRebY8%2BvAawbQCVRklayH56LIscFxRRl2WhtSNimnpFlMWu7hKoiK5iOMoDPwhsxAd94aMRvTi%2F6Yx8z9yTwv402ayytaqFuzNuVG6oebzyAIvGF9E6VYjlPCGinpZlpoD2OjqWvXXmlNj99zojiN%2FcVT9d9MX7w%3D%3D&RelayState=ver%3A3-hint%3A260092117886646-ETMsDgAAAZ7QLr7vABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEMM9OcR6ZCYvYCXRu81a4u0AAACQ%2Fzx9iPk5LmqQjZ

In [12]:
# =============================================================================
# AUTH PULL: case-level detail for flagged hospital_groups
# notifications_date = 06102026 (latest available)
# Filters: MNR_TOTAL_FFS_FLAG = 1, LOC_FLAG = 1
# =============================================================================
flagged_hospitals = flagged["HOSPITAL_GROUP"].dropna().unique().tolist()
placeholders = ",".join([f"'{h}'" for h in flagged_hospitals])

df_auth = pd.read_sql(f"""
    select
        d.collection as hospital_group
        , a.*
    from ving_prd_trend_db.tmp_1m.ec_ip_dataset_06102026_3_od as a
    left join ving_prd_trend_db.tmp_1y.tin_collection as d
        on a.prov_tin = d.tin
    where d.collection in ({placeholders})
        and a.mnr_total_ffs_flag = 1
        and a.loc_flag = 1
""", engine_write)

engine_write.dispose()
print(f"Auth pull: {df_auth.shape[0]} rows, {df_auth.shape[1]} cols for {df_auth['hospital_group'].nunique()} hospital groups")
df_auth.head()

Auth pull: 748517 rows, 80 cols for 44 hospital groups


,hospital_group,admit_week,hce_dt,hce_admit_month,admit_act_month,admit_act_qtr,admit_year,admit_dt_act,admit_dt_exp,dschg_dt_act,...,p2p_case_cnt,p2p_ovrtn_case_cnt,member_appeal_cnt,member_appeal_ovtn_cnt,other_ovtrns,membership,swgbed,mr_cs_other,admit_type,ipa_pac_flag
0,Novant Health System,202532,2025-08-05,202508,202508,2025Q3,2025,2025-08-05,2025-08-05,2025-08-05 00:00:00,...,0,0,0,0,0,0,0,M&R,Medical,IPA
1,HCA INC,202602,2026-01-09,202601,202601,2026Q1,2026,2026-01-09,2026-01-09,2026-01-20 00:00:00,...,1,0,0,0,0,0,0,M&R,Medical,IPA
2,HCA INC,202609,2026-02-23,202602,202602,2026Q1,2026,2026-02-23,2026-02-23,2026-03-01 00:00:00,...,0,0,0,0,0,0,0,M&R,Medical,IPA
3,Adventist Health System Florida,202540,2025-10-03,202510,202510,2025Q4,2025,2025-10-03,2025-10-03,2025-10-10 00:00:00,...,0,0,0,0,0,0,0,M&R,Medical,IPA
4,ADVOCATE HEALTH & HOSPITALS IL,202552,2025-12-22,202512,202512,2025Q4,2025,2025-12-22,2025-12-22,2026-01-05 00:00:00,...,0,0,0,0,0,0,0,M&R,Medical,IPA
